# Algorithm versus physician analysis
Runs the analysis for the VUmc, LUMC, and combined enrolled cohorts.

In [ ]:
# Parameters

project_root = "../"
data_dir = "../data"
n_bootstraps = 10000
random_seed = 42

# Settings and Data

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import algo_vs_doc.analysis as analysis
from algo_vs_doc import (
    baseline_descriptives,
    bootstrap_storage,
    low_risk_event_tables,
    shared_bootstrap,
    clinical_performance,
    cohorts,
    comparison,
    prediction_data,
    risk_factors,
)
from algo_vs_doc.scoring.final_assembly import (
    OUTCOME_ELIGIBILITY_STEPS,
    apply_enrolment_funnel,
)

pd.set_option("mode.chained_assignment", None)
pd.set_option("display.max_columns", None)

outputs_folder = os.path.join(project_root, "results", "summaries")
plot_data_folder = os.path.join(project_root, "results", "plot_data")
for name in cohorts.COHORT_ORDER:
    os.makedirs(os.path.join(plot_data_folder, name), exist_ok=True)

os.makedirs(outputs_folder, exist_ok=True)
for name in cohorts.COHORT_ORDER:
    os.makedirs(os.path.join(outputs_folder, name), exist_ok=True)
bootstrap_storage.configure(
    os.path.join(project_root, "results", "bootstrap"),
    random_seed=random_seed,
    n_bootstraps=n_bootstraps,
)
# Reset only when starting a new analysis run, not between metric cells.
bootstrap_run = shared_bootstrap.begin_run()

## predictions_and_outcomes
- each row corresponds to one prediction of algorithm and doc
- contains the outcome (readmision or death)

## Transform data and combine into single dataframe

In [ ]:
loaded_sites = prediction_data.load_cohorts(data_dir, include_combined=False)
outcome_eligible_sites = {
    name: apply_enrolment_funnel(frame, steps=OUTCOME_ELIGIBILITY_STEPS)[0]
    for name, frame in loaded_sites.items()
}
outcome_eligible_cohorts = {
    **outcome_eligible_sites,
    "Combined": cohorts.combine(outcome_eligible_sites.values()),
}
enrolled_sites = {
    name: apply_enrolment_funnel(frame)[0] for name, frame in loaded_sites.items()
}
loaded_cohorts = {
    "VUmc": enrolled_sites["VUmc"],
    "LUMC": enrolled_sites["LUMC"],
    "Combined": cohorts.combine(enrolled_sites.values()),
}
for frame in [*loaded_cohorts.values(), *outcome_eligible_cohorts.values()]:
    frame["patient_cluster_id"] = (
        frame["hospital"].astype(str) + "::" + frame["patient_id"].astype(str)
    )
for frame in loaded_cohorts.values():
    frame["doc_prediction_cat_score"] = frame["doc_prediction_cat"].map(
        {"low": 0.0, "medium": 1.0, "high": 2.0}
    )

pd.Series(
    {name: len(frame) for name, frame in loaded_cohorts.items()},
    name="predictions",
).to_frame()

## Clinically interpretable operating points

These supplementary analyses address performance under explicit intervention budgets. Fixed alert rates rank discharges by predicted risk and therefore do not define a transportable probability threshold. The Youden threshold is estimated and evaluated in this same small cohort, so it is descriptive and optimistic rather than a proposed clinical decision rule.


In [ ]:
configured_threshold_rows = []
for cohort_name, cohort_data in loaded_cohorts.items():
    prediction_column = "algo_prediction_at_doc_prediction"
    operating_points = [
        ("5% risk threshold", 0.05),
        ("10% risk threshold", 0.10),
        ("20% risk threshold", 0.20),
        (
            "Youden index",
            clinical_performance.youden_threshold(
                cohort_data, prediction=prediction_column, outcome="outcome"
            ),
        ),
    ]
    for operating_point, threshold in operating_points:
        configured_threshold_rows.append(
            {
                "cohort": cohort_name,
                "prediction_source": "Model at physician prediction",
                "operating_point": operating_point,
                **clinical_performance.threshold_metrics(
                    cohort_data,
                    prediction=prediction_column,
                    outcome="outcome",
                    threshold=threshold,
                    cluster_column="patient_cluster_id",
                    n_bootstraps=n_bootstraps,
                    progress_desc=f"{cohort_name}: Model: {operating_point}",
                    random_seed=random_seed,
                ),
            }
        )
configured_threshold_performance = pd.DataFrame(configured_threshold_rows)
configured_threshold_performance.to_csv(
    os.path.join(outputs_folder, "configured_threshold_performance.csv"),
    index=False,
)
configured_threshold_performance

In [ ]:
os.makedirs(outputs_folder, exist_ok=True)
prediction_sources = {
    "Physician": "doc_prediction",
    "Model at physician prediction": "algo_prediction_at_doc_prediction",
    "Model at discharge": "algo_prediction_at_discharge",
}
n_bootstrap_analyses = (
    len(loaded_cohorts) * (2 * len(prediction_sources))
    + len(outcome_eligible_cohorts) * 2
)
overall_progress = tqdm(
    total=n_bootstrap_analyses,
    desc="All bootstrap analyses",
    unit="analysis",
)


def track_bootstrap_analyses(items):
    for item in items:
        yield item
        overall_progress.update()


alert_rate_rows = []
threshold_rows = []
for cohort_name, cohort_data in loaded_cohorts.items():
    for source, prediction_column in prediction_sources.items():
        alert_rate_rows.append(
            clinical_performance.alert_rate_table(
                cohort_data,
                prediction=prediction_column,
                outcome="outcome",
                cluster_column="patient_cluster_id",
                n_bootstraps=n_bootstraps,
                progress_desc=f"{cohort_name}: {source}: alert rates",
                random_seed=random_seed,
            ).assign(cohort=cohort_name, prediction_source=source)
        )
        overall_progress.update()
        threshold = clinical_performance.youden_threshold(
            cohort_data, prediction=prediction_column, outcome="outcome"
        )
        threshold_rows.append(
            {
                "cohort": cohort_name,
                "prediction_source": source,
                **clinical_performance.threshold_metrics(
                    cohort_data,
                    prediction=prediction_column,
                    outcome="outcome",
                    threshold=threshold,
                    cluster_column="patient_cluster_id",
                    n_bootstraps=n_bootstraps,
                    progress_desc=f"{cohort_name}: {source}: Youden metrics",
                    random_seed=random_seed,
                ),
            }
        )
        overall_progress.update()

alert_rates = pd.concat(alert_rate_rows, ignore_index=True)
threshold_performance = pd.DataFrame(threshold_rows)

full_eligible_discharge_alert_rates = pd.concat(
    [
        clinical_performance.alert_rate_table(
            cohort_data,
            prediction="algo_prediction_at_discharge",
            outcome="outcome",
            cluster_column="patient_cluster_id",
            n_bootstraps=n_bootstraps,
            progress_desc=f"{cohort_name}: full eligible: alert rates",
            random_seed=random_seed,
        ).assign(cohort=cohort_name, prediction_source="Model at discharge")
        for cohort_name, cohort_data in track_bootstrap_analyses(
            outcome_eligible_cohorts.items()
        )
    ],
    ignore_index=True,
)

alert_rates.to_csv(
    os.path.join(outputs_folder, "fixed_alert_rate_performance.csv"), index=False
)
threshold_performance.to_csv(
    os.path.join(outputs_folder, "exploratory_youden_threshold_performance.csv"),
    index=False,
)
full_eligible_discharge_performance = pd.concat(
    [
        analysis.performance_table(
            predictions=cohort_data,
            outcome="outcome",
            prediction_columns={"Model at discharge": "algo_prediction_at_discharge"},
            curve="roc",
            n_bootstraps=n_bootstraps,
            cluster_column="patient_cluster_id",
            progress_desc=f"{cohort_name}: full eligible AUROC",
            random_seed=random_seed,
        )
        .rename_axis("prediction_source")
        .reset_index()
        .assign(cohort=cohort_name)
        for cohort_name, cohort_data in track_bootstrap_analyses(
            outcome_eligible_cohorts.items()
        )
    ],
    ignore_index=True,
)
full_eligible_discharge_alert_rates.to_csv(
    os.path.join(outputs_folder, "full_eligible_model_at_discharge_alert_rates.csv"),
    index=False,
)
full_eligible_discharge_performance.to_csv(
    os.path.join(outputs_folder, "full_eligible_model_at_discharge_performance.csv"),
    index=False,
)
overall_progress.close()
alert_rates

## Events observed in physician risk categories

Only the physician supplied a prespecified low, medium, or high category. Model and combined categories are not manufactured from arbitrary cutoffs; they can be added when a defensible prespecified definition is provided.


In [ ]:
risk_category_events = pd.concat(
    [
        clinical_performance.risk_category_events(
            cohort_data,
            category_columns={"Physician": "doc_prediction_cat"},
            outcome="outcome",
        ).assign(cohort=cohort_name)
        for cohort_name, cohort_data in loaded_cohorts.items()
    ],
    ignore_index=True,
)
risk_category_events.to_csv(
    os.path.join(outputs_folder, "risk_category_events.csv"), index=False
)
risk_category_events

## Adverse events assigned low risk

These aggregate summaries describe adverse-event cases assigned the physician's prespecified low category or falling below the model's cohort-derived operating threshold. They contain no patient identifiers and cannot determine from structured data whether a readmission was clinically unrelated to the index stay.


In [ ]:
low_risk_event_rows = []
for cohort_name, cohort_data in loaded_cohorts.items():
    low_risk_definitions = {
        "Physician low category": cohort_data["doc_prediction_cat"].eq("low"),
    }
    low_risk_definitions["Model predicted risk <5%"] = cohort_data[
        "algo_prediction_at_doc_prediction"
    ].lt(0.05)
    for definition, low_risk in low_risk_definitions.items():
        cases = cohort_data.loc[low_risk & cohort_data["outcome"].astype(bool)]
        event_components = pd.Series(
            {
                "Low-risk events, n": len(cases),
                "ICU readmission within 7 days, n": int(
                    cases["has_icu_readmission_7days"].astype(bool).sum()
                ),
                "Death within 7 days, n": int(
                    cases["has_death_7days"].astype(bool).sum()
                ),
                "ICU readmission and death within 7 days, n": int(
                    (
                        cases["has_icu_readmission_7days"].astype(bool)
                        & cases["has_death_7days"].astype(bool)
                    ).sum()
                ),
            }
        )
        summary = pd.concat(
            [event_components, baseline_descriptives.extended_context_summary(cases)]
        )
        low_risk_event_rows.append(
            summary.rename("value")
            .rename_axis("characteristic")
            .reset_index()
            .assign(cohort=cohort_name, low_risk_definition=definition)
        )

low_risk_event_characteristics = pd.concat(low_risk_event_rows, ignore_index=True)[
    ["cohort", "low_risk_definition", "characteristic", "value"]
]
low_risk_event_characteristics.to_csv(
    os.path.join(outputs_folder, "low_risk_event_characteristics.csv"),
    index=False,
)
low_risk_event_characteristics


individual_case_dir = os.path.join(project_root, "results", "individual_cases")
os.makedirs(individual_case_dir, exist_ok=True)
low_risk_event_tables.individual_event_entries(loaded_cohorts).to_csv(
    os.path.join(individual_case_dir, "low_risk_event_entries.csv"), index=False
)

# Exploratory data analysis


## Correlation

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = loaded_cohorts[cohort]
    correlations = predictions_and_outcomes[
        [
            "algo_prediction_at_doc_prediction",
            "algo_prediction_at_discharge",
            "doc_prediction",
        ]
    ].corr("spearman")
    correlations.to_csv(os.path.join(outputs_folder, cohort, "correlation_table.csv"))

## Calculate the combined prediction of physicians and algorithm

In [ ]:
analysis_cohorts = {}
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = loaded_cohorts[cohort].copy()
    predictions_and_outcomes = analysis.combine_predictions(
        predictions_and_outcomes,
        "doc_prediction",
        "algo_prediction_at_doc_prediction",
        "outcome",
        "algo_and_doc_prediction",
        group_column="patient_cluster_id",
        random_seed=random_seed,
    )
    predictions_and_outcomes = analysis.combine_predictions(
        predictions_and_outcomes,
        "doc_prediction_cat",
        "algo_prediction_at_doc_prediction",
        "outcome",
        "algo_and_doc_cat_prediction",
        group_column="patient_cluster_id",
        prediction1_is_ordinal=True,
        random_seed=random_seed,
    )
    analysis_cohorts[cohort] = predictions_and_outcomes

## Discrimination and calibration of predictions

## Individual predictor performance

In [ ]:
analysis_states = {}
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    table = analysis.performance_table(
        predictions=predictions_and_outcomes,
        outcome="outcome",
        prediction_columns={
            "Physician (Percentage)": "doc_prediction",
            "Physician (Category)": "doc_prediction_cat_score",
            "Model at Physician Prediction": "algo_prediction_at_doc_prediction",
            "Model at Patient Discharge": "algo_prediction_at_discharge",
            "Combined (Percentage)": "algo_and_doc_prediction",
            "Combined (Category)": "algo_and_doc_cat_prediction",
        },
        curve="roc",
        n_bootstraps=n_bootstraps,
        cluster_column="patient_cluster_id",
        ordinal_predictions={"Physician (Category)"},
        progress_desc=f"{cohort}: predictor performance",
        random_seed=random_seed,
    )
    combined_specifications = {
        "Combined (Percentage)": {
            "prediction1": "doc_prediction",
            "combined_prediction": "algo_and_doc_prediction",
            "comparisons": {
                "Physician (Percentage)": "doc_prediction",
                "Model at Physician Prediction": "algo_prediction_at_doc_prediction",
            },
        },
        "Combined (Category)": {
            "prediction1": "doc_prediction_cat",
            "prediction1_is_ordinal": True,
            "combined_prediction": "algo_and_doc_cat_prediction",
            "comparisons": {
                "Physician (Category)": "doc_prediction_cat_score",
                "Model at Physician Prediction": "algo_prediction_at_doc_prediction",
            },
        },
    }
    combined_comparison_rows = []
    combined_alert_rate_rows = []
    auroc_contrast_rows = []
    auroc_bootstrap_rows = []

    analysis_states[cohort] = {
        "table": table,
        "combined_specifications": combined_specifications,
        "combined_comparison_rows": combined_comparison_rows,
        "combined_alert_rate_rows": combined_alert_rate_rows,
        "auroc_contrast_rows": auroc_contrast_rows,
        "auroc_bootstrap_rows": auroc_bootstrap_rows,
    }

## Percentage-based combined prediction

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    state = analysis_states[cohort]
    table = state["table"]
    combined_specifications = state["combined_specifications"]
    combined_comparison_rows = state["combined_comparison_rows"]
    combined_alert_rate_rows = state["combined_alert_rate_rows"]
    auroc_contrast_rows = state["auroc_contrast_rows"]
    auroc_bootstrap_rows = state["auroc_bootstrap_rows"]
    combined_label = "Combined (Percentage)"
    specification = combined_specifications[combined_label]
    nested_performance, nested_comparisons, nested_alert_rates, nested_samples = (
        analysis.nested_bootstrap_combined_auc(
            predictions_and_outcomes,
            prediction1=specification["prediction1"],
            prediction2="algo_prediction_at_doc_prediction",
            outcome="outcome",
            combined_prediction=specification["combined_prediction"],
            comparison_predictions=specification["comparisons"],
            group_column="patient_cluster_id",
            n_bootstraps=n_bootstraps,
            curve="roc",
            prediction1_is_ordinal=specification.get("prediction1_is_ordinal", False),
            progress_desc=f"{cohort}: {combined_label}",
            random_seed=random_seed,
        )
    )
    for calibration_key, calibration_value in nested_performance.items():
        if calibration_key.startswith(
            ("calibration_", "observed_expected_ratio", "brier_score")
        ):
            table.loc[combined_label, calibration_key] = calibration_value
    table.loc[combined_label, "auroc"] = nested_performance["auroc"]
    table.loc[combined_label, "auroc_confidence_interval_lower"] = nested_performance[
        "auroc_confidence_interval_lower"
    ]
    table.loc[combined_label, "auroc_confidence_interval_upper"] = nested_performance[
        "auroc_confidence_interval_upper"
    ]
    table.loc[combined_label, "auroc_with_confidence_interval"] = (
        f"{nested_performance['auroc']:#.2g} "
        f"[{nested_performance['auroc_confidence_interval_lower']:#.2g}, "
        f"{nested_performance['auroc_confidence_interval_upper']:#.2g}]"
    )
    table.loc[combined_label, "auprc"] = nested_performance["auprc"]
    table.loc[combined_label, "auprc_confidence_interval_lower"] = nested_performance[
        "auprc_confidence_interval_lower"
    ]
    table.loc[combined_label, "auprc_confidence_interval_upper"] = nested_performance[
        "auprc_confidence_interval_upper"
    ]
    table.loc[combined_label, "auprc_with_confidence_interval"] = (
        f"{nested_performance['auprc']:#.2g} "
        f"[{nested_performance['auprc_confidence_interval_lower']:#.2g}, "
        f"{nested_performance['auprc_confidence_interval_upper']:#.2g}]"
    )
    table.loc[combined_label, "valid_bootstraps"] = nested_performance[
        "valid_bootstraps"
    ]
    combined_comparison_rows.append(nested_comparisons.assign(agent_1=combined_label))
    auroc_bootstrap_rows.append(nested_samples.assign(agent_1=combined_label))
    combined_alert_rate_rows.append(
        nested_alert_rates.assign(cohort=cohort, prediction_source=combined_label)
    )

## Category-based combined prediction

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    state = analysis_states[cohort]
    table = state["table"]
    combined_specifications = state["combined_specifications"]
    combined_comparison_rows = state["combined_comparison_rows"]
    combined_alert_rate_rows = state["combined_alert_rate_rows"]
    auroc_contrast_rows = state["auroc_contrast_rows"]
    auroc_bootstrap_rows = state["auroc_bootstrap_rows"]
    combined_label = "Combined (Category)"
    specification = combined_specifications[combined_label]
    nested_performance, nested_comparisons, nested_alert_rates, nested_samples = (
        analysis.nested_bootstrap_combined_auc(
            predictions_and_outcomes,
            prediction1=specification["prediction1"],
            prediction2="algo_prediction_at_doc_prediction",
            outcome="outcome",
            combined_prediction=specification["combined_prediction"],
            comparison_predictions=specification["comparisons"],
            group_column="patient_cluster_id",
            n_bootstraps=n_bootstraps,
            curve="roc",
            prediction1_is_ordinal=specification.get("prediction1_is_ordinal", False),
            progress_desc=f"{cohort}: {combined_label}",
            random_seed=random_seed,
        )
    )
    for calibration_key, calibration_value in nested_performance.items():
        if calibration_key.startswith(
            ("calibration_", "observed_expected_ratio", "brier_score")
        ):
            table.loc[combined_label, calibration_key] = calibration_value
    table.loc[combined_label, "auroc"] = nested_performance["auroc"]
    table.loc[combined_label, "auroc_confidence_interval_lower"] = nested_performance[
        "auroc_confidence_interval_lower"
    ]
    table.loc[combined_label, "auroc_confidence_interval_upper"] = nested_performance[
        "auroc_confidence_interval_upper"
    ]
    table.loc[combined_label, "auroc_with_confidence_interval"] = (
        f"{nested_performance['auroc']:#.2g} "
        f"[{nested_performance['auroc_confidence_interval_lower']:#.2g}, "
        f"{nested_performance['auroc_confidence_interval_upper']:#.2g}]"
    )
    table.loc[combined_label, "auprc"] = nested_performance["auprc"]
    table.loc[combined_label, "auprc_confidence_interval_lower"] = nested_performance[
        "auprc_confidence_interval_lower"
    ]
    table.loc[combined_label, "auprc_confidence_interval_upper"] = nested_performance[
        "auprc_confidence_interval_upper"
    ]
    table.loc[combined_label, "auprc_with_confidence_interval"] = (
        f"{nested_performance['auprc']:#.2g} "
        f"[{nested_performance['auprc_confidence_interval_lower']:#.2g}, "
        f"{nested_performance['auprc_confidence_interval_upper']:#.2g}]"
    )
    table.loc[combined_label, "valid_bootstraps"] = nested_performance[
        "valid_bootstraps"
    ]
    combined_comparison_rows.append(nested_comparisons.assign(agent_1=combined_label))
    auroc_bootstrap_rows.append(nested_samples.assign(agent_1=combined_label))
    combined_alert_rate_rows.append(
        nested_alert_rates.assign(cohort=cohort, prediction_source=combined_label)
    )

### Differences in model performance

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    state = analysis_states[cohort]
    table = state["table"]
    combined_specifications = state["combined_specifications"]
    combined_comparison_rows = state["combined_comparison_rows"]
    combined_alert_rate_rows = state["combined_alert_rate_rows"]
    auroc_contrast_rows = state["auroc_contrast_rows"]
    auroc_bootstrap_rows = state["auroc_bootstrap_rows"]
    primary_auroc_table, data = analysis.auc_differences_table(
        predictions=predictions_and_outcomes,
        outcome="outcome",
        prediction_columns={
            "Model": "algo_prediction_at_doc_prediction",
            "Physician (Percentage)": "doc_prediction",
        },
        curve="roc",
        n_bootstraps=n_bootstraps,
        cluster_column="patient_cluster_id",
        adjust_holm=False,
        progress_desc=f"{cohort}: primary AUROC comparisons",
        random_seed=random_seed,
    )
    primary_auroc_table.to_csv(
        os.path.join(outputs_folder, cohort, "primary_auroc_difference.csv")
    )
    primary_comparison = data["Model"]["Physician (Percentage)"]
    primary_bootstrap_plot_data = pd.DataFrame(
        {
            "bootstrap_difference_agent_1_minus_agent_2": primary_comparison[4],
            "observed_difference": primary_comparison[0],
            "confidence_interval_lower": primary_comparison[1],
            "confidence_interval_upper": primary_comparison[2],
            "p": primary_comparison[3],
            "curve": primary_comparison[5],
        }
    )
    primary_bootstrap_plot_data.to_csv(
        os.path.join(
            plot_data_folder, cohort, "primary_auroc_difference_plot_data.csv"
        ),
        index=False,
    )
    primary_auroc_table
    auroc_contrast_rows.append(
        {
            "agent_1": "Model at Physician Prediction",
            "agent_2": "Physician (Percentage)",
            "difference_agent_1_minus_agent_2": primary_comparison[0],
            "confidence_interval_lower": primary_comparison[1],
            "confidence_interval_upper": primary_comparison[2],
            "p": primary_comparison[3],
            "valid_bootstraps": len(primary_comparison[4]),
        }
    )
    auroc_bootstrap_rows.append(
        pd.DataFrame(
            {
                "agent_1": "Model at Physician Prediction",
                "agent_2": "Physician (Percentage)",
                "bootstrap_difference_agent_1_minus_agent_2": primary_comparison[4],
            }
        )
    )

## Secondary performance comparisons

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    state = analysis_states[cohort]
    table = state["table"]
    combined_specifications = state["combined_specifications"]
    combined_comparison_rows = state["combined_comparison_rows"]
    combined_alert_rate_rows = state["combined_alert_rate_rows"]
    auroc_contrast_rows = state["auroc_contrast_rows"]
    auroc_bootstrap_rows = state["auroc_bootstrap_rows"]
    category_complete = predictions_and_outcomes[
        [
            "outcome",
            "doc_prediction",
            "doc_prediction_cat_score",
            "algo_prediction_at_doc_prediction",
            "patient_cluster_id",
        ]
    ].dropna()
    model_category_difference = analysis.bootstrap_AUC_difference(
        n_bootstraps,
        category_complete["outcome"],
        category_complete["algo_prediction_at_doc_prediction"],
        category_complete["doc_prediction_cat_score"],
        curve="roc",
        clusters=category_complete["patient_cluster_id"],
        progress_desc=f"{cohort}: model vs physician category",
        random_seed=random_seed,
    )
    combined_comparison_rows.append(
        pd.DataFrame(
            [
                {
                    "agent_1": "Model at Physician Prediction",
                    "agent_2": "Physician (Category)",
                    "difference_agent_1_minus_agent_2": model_category_difference[0],
                    "confidence_interval_lower": model_category_difference[1],
                    "confidence_interval_upper": model_category_difference[2],
                    "p": model_category_difference[3],
                    "valid_bootstraps": len(model_category_difference[4]),
                }
            ]
        )
    )
    elicitation_difference = analysis.bootstrap_AUC_difference(
        n_bootstraps,
        category_complete["outcome"],
        category_complete["doc_prediction_cat_score"],
        category_complete["doc_prediction"],
        curve="roc",
        clusters=category_complete["patient_cluster_id"],
        progress_desc=f"{cohort}: physician percentage vs category",
        random_seed=random_seed,
    )
    combined_comparison_rows.append(
        pd.DataFrame(
            [
                {
                    "agent_1": "Physician (Category)",
                    "agent_2": "Physician (Percentage)",
                    "difference_agent_1_minus_agent_2": elicitation_difference[0],
                    "confidence_interval_lower": elicitation_difference[1],
                    "confidence_interval_upper": elicitation_difference[2],
                    "p": elicitation_difference[3],
                    "valid_bootstraps": len(elicitation_difference[4]),
                }
            ]
        )
    )
    model_category_row = combined_comparison_rows[-2].iloc[0].to_dict()
    elicitation_row = combined_comparison_rows[-1].iloc[0].to_dict()
    auroc_contrast_rows.extend([model_category_row, elicitation_row])
    auroc_bootstrap_rows.extend(
        [
            pd.DataFrame(
                {
                    "agent_1": "Model at Physician Prediction",
                    "agent_2": "Physician (Category)",
                    "bootstrap_difference_agent_1_minus_agent_2": model_category_difference[
                        4
                    ],
                }
            ),
            pd.DataFrame(
                {
                    "agent_1": "Physician (Category)",
                    "agent_2": "Physician (Percentage)",
                    "bootstrap_difference_agent_1_minus_agent_2": elicitation_difference[
                        4
                    ],
                }
            ),
        ]
    )
    auroc_contrasts = pd.DataFrame(
        auroc_contrast_rows
        + [
            row
            for frame in combined_comparison_rows[:-2]
            for row in frame.to_dict("records")
        ]
    )
    model_label = "Model at Physician Prediction"
    auroc_contrasts["family"] = "exploratory"
    primary = auroc_contrasts["agent_1"].eq(model_label) & auroc_contrasts[
        "agent_2"
    ].isin(["Physician (Percentage)", "Physician (Category)"])
    auroc_contrasts.loc[primary, "family"] = "primary"
    auroc_contrasts["p_holm"] = np.nan
    auroc_contrasts.loc[primary, "p_holm"] = comparison.holm_adjust(
        auroc_contrasts.loc[primary, "p"]
    )
    auroc_contrasts["contrast_id"] = np.arange(len(auroc_contrasts))
    contrast_ids = auroc_contrasts.set_index(["agent_1", "agent_2"])["contrast_id"]
    auroc_bootstraps = pd.concat(auroc_bootstrap_rows, ignore_index=True)
    auroc_bootstraps["contrast_id"] = [
        contrast_ids.loc[(agent_1, agent_2)]
        for agent_1, agent_2 in zip(
            auroc_bootstraps["agent_1"],
            auroc_bootstraps["agent_2"],
        )
    ]
    auroc_contrasts["metric"] = "auroc"
    exploratory_auroc_differences = auroc_contrasts.loc[
        auroc_contrasts["family"].eq("exploratory")
    ].copy()
    alert_rates = pd.concat([alert_rates, *combined_alert_rate_rows], ignore_index=True)
    table.to_csv(os.path.join(outputs_folder, cohort, "performance_table.csv"))
    auroc_contrasts.to_csv(
        os.path.join(outputs_folder, cohort, "auroc_contrasts.csv"),
        index=False,
    )
    auroc_bootstraps.to_csv(
        os.path.join(plot_data_folder, cohort, "auroc_contrast_bootstraps.csv"),
        index=False,
    )
    exploratory_auroc_differences.to_csv(
        os.path.join(outputs_folder, cohort, "exploratory_auroc_differences.csv"),
        index=False,
    )
    alert_rates.to_csv(
        os.path.join(outputs_folder, "fixed_alert_rate_performance.csv"), index=False
    )
    table
    state["auroc_contrasts"] = auroc_contrasts
    state["auroc_bootstraps"] = auroc_bootstraps

## Additional exploratory pairwise comparisons
Three additional AUROC contrasts and all ten AUPRC contrasts. Both combination models are refitted on shared patient bootstrap samples. AUPRC is calculated as average precision. The two model-versus-physician AUPRC contrasts form a secondary family with Holm adjustment. Remaining new comparisons are exploratory with unadjusted p-values; primary AUROC inference is unchanged.

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    state = analysis_states[cohort]
    auroc_contrasts = state["auroc_contrasts"]
    auroc_bootstraps = state["auroc_bootstraps"]
    from algo_vs_doc import exploratory_contrasts

    exploratory_rows, exploratory_samples = (
        exploratory_contrasts.supplementary_contrasts(
            predictions_and_outcomes,
            n_bootstraps=n_bootstraps,
            random_seed=random_seed,
        )
    )
    for metric in ("auroc", "auprc"):
        rows = exploratory_rows.loc[exploratory_rows["metric"].eq(metric)].copy()
        samples = exploratory_samples.loc[
            exploratory_samples["metric"].eq(metric)
        ].copy()
        if metric == "auroc":
            # Retain the existing primary estimates without recalculation.
            existing = auroc_contrasts.copy()
            existing_samples = auroc_bootstraps.loc[
                auroc_bootstraps["contrast_id"].isin(existing["contrast_id"])
            ].copy()
            offset = int(existing["contrast_id"].max()) + 1
            rows["contrast_id"] += offset
            samples["contrast_id"] += offset
            rows = pd.concat([existing, rows], ignore_index=True)
            samples = pd.concat([existing_samples, samples], ignore_index=True)
            auroc_contrasts, auroc_bootstraps = rows, samples
            rows.to_csv(
                os.path.join(outputs_folder, cohort, "auroc_contrasts.csv"), index=False
            )
            samples.to_csv(
                os.path.join(plot_data_folder, cohort, "auroc_contrast_bootstraps.csv"),
                index=False,
            )
            rows.loc[rows["family"].eq("exploratory")].to_csv(
                os.path.join(
                    outputs_folder, cohort, "exploratory_auroc_differences.csv"
                ),
                index=False,
            )
        else:
            rows.to_csv(
                os.path.join(outputs_folder, cohort, "auprc_contrasts.csv"), index=False
            )
            samples.to_csv(
                os.path.join(plot_data_folder, cohort, "auprc_contrast_bootstraps.csv"),
                index=False,
            )

## AUC with different time thresholds

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    time_dif_data = analysis.create_max_time_difference_df(
        predictions_all_time_points=predictions_and_outcomes,
        outcome="outcome",
        prediction_columns={
            "Physician (Percentage)": "doc_prediction",
            "Physician (Category)": "doc_prediction_cat_score",
            "Model": "algo_prediction_at_doc_prediction",
        },
        curve="roc",
        n_bootstraps=n_bootstraps,
        cluster_column="patient_cluster_id",
        progress_desc=f"{cohort}: prediction timing sensitivity",
        random_seed=random_seed,
    )
    time_dif_data.to_csv(
        os.path.join(outputs_folder, cohort, "prediction_timing_sensitivity.csv"),
        index=False,
    )

## Confidence level

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    performance_table_high_confidence = analysis.performance_table(
        predictions=predictions_and_outcomes.loc[
            predictions_and_outcomes.confidence == "high", :
        ],
        outcome="outcome",
        prediction_columns={
            "Physician (Percentage)": "doc_prediction",
            "Physician (Category)": "doc_prediction_cat_score",
            "Model": "algo_prediction_at_doc_prediction",
        },
        curve="roc",
        n_bootstraps=n_bootstraps,
        cluster_column="patient_cluster_id",
        ordinal_predictions={"Physician (Category)"},
        progress_desc=f"{cohort}: high-confidence performance",
        random_seed=random_seed,
    )
    performance_table_high_confidence.to_csv(
        os.path.join(outputs_folder, cohort, "performance_table_high_confidence.csv")
    )

    performance_table_medium_low_confidence = analysis.performance_table(
        predictions=predictions_and_outcomes.loc[
            predictions_and_outcomes.confidence == "low/medium", :
        ],
        outcome="outcome",
        prediction_columns={
            "Physician (Percentage)": "doc_prediction",
            "Physician (Category)": "doc_prediction_cat_score",
            "Model": "algo_prediction_at_doc_prediction",
        },
        curve="roc",
        n_bootstraps=n_bootstraps,
        cluster_column="patient_cluster_id",
        ordinal_predictions={"Physician (Category)"},
        progress_desc=f"{cohort}: lower-confidence performance",
        random_seed=random_seed,
    )
    performance_table_medium_low_confidence.to_csv(
        os.path.join(
            outputs_folder,
            cohort,
            "performance_table_medium_low_confidence.csv",
        )
    )

## Comparison factors for algo and doc

In [ ]:
feature_states = {}
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    # Extract model features and stay keys.
    shap_variable = predictions_and_outcomes.features_at_doc_prediction
    patient_id_variable = predictions_and_outcomes.patient_id
    icu_discharged_at_variable = predictions_and_outcomes.icu_discharged_at
    feature_states[cohort] = {
        "shap_variable": shap_variable,
        "patient_id_variable": patient_id_variable,
        "icu_discharged_at_variable": icu_discharged_at_variable,
    }

### Algorithm factors (model features)

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    state = feature_states[cohort]
    shap_variable = state["shap_variable"]
    patient_id_variable = state["patient_id_variable"]
    icu_discharged_at_variable = state["icu_discharged_at_variable"]
    # Get model factors based on Shapley values
    shap_data = analysis.construct_shap_data_frame(
        shap_variable, patient_id_variable, icu_discharged_at_variable
    )

    algo_factors = shap_data.pivot_table(
        index=["patient_id", "icu_discharged_at"],
        columns=["feature_name"],
        values=["shap_value"],
        aggfunc=np.size,
    )

    algo_factors_summary = (
        pd.DataFrame((~algo_factors.isnull()).sum())
        .reset_index()
        .rename(columns={0: "count", "feature_name": "factor"})[["factor", "count"]]
        .sort_values(by=["count", "factor"], ascending=[False, True])
    )

    algo_factors_summary.to_csv(
        os.path.join(outputs_folder, cohort, "algo_factors.csv"), index=False
    )

    # Get model factors with new categories
    shap_data_new = analysis.get_new_algo_features(
        shap_data, limit=3, absolute_values=True
    )
    algo_factors_new = pd.pivot(
        shap_data_new,
        index=["patient_id", "icu_discharged_at"],
        columns=["new_feature_name"],
        values=["shap_value"],
    )

    algo_factors_new_summary = (
        pd.DataFrame((~algo_factors_new.isnull()).sum())
        .reset_index()
        .rename(columns={0: "count", "new_feature_name": "factor_category"})[
            ["factor_category", "count"]
        ]
        .sort_values(by=["count", "factor_category"], ascending=[False, True])
    )

    algo_factors_new_summary["percentage"] = round(
        100
        * algo_factors_new_summary["count"]
        / algo_factors_new_summary["count"].sum(),
        1,
    )

    algo_factors_new_summary.to_csv(
        os.path.join(outputs_folder, cohort, "algo_factor_categories.csv"),
        index=False,
    )

    shap_with_hospital = shap_data_new.merge(
        predictions_and_outcomes[
            ["patient_id", "icu_discharged_at", "hospital"]
        ].drop_duplicates(),
        on=["patient_id", "icu_discharged_at"],
        how="left",
        validate="many_to_one",
    )
    shap_with_hospital["feature_label"] = risk_factors.model_feature_labels(
        shap_with_hospital["feature_name"]
    )
    model_feature_mapping = (
        shap_with_hospital.groupby(
            ["new_feature_name", "feature_label", "hospital"], dropna=False
        )
        .size()
        .unstack("hospital", fill_value=0)
        .reindex(columns=cohorts.HOSPITALS, fill_value=0)
        .rename_axis(index=["category", "model_feature"], columns=None)
        .reset_index()
    )
    model_feature_mapping["category"] = pd.Categorical(
        model_feature_mapping["category"],
        categories=risk_factors.CATEGORIES,
        ordered=True,
    )
    model_feature_mapping = model_feature_mapping.sort_values(
        ["category", "model_feature"]
    ).reset_index(drop=True)
    model_feature_mapping = model_feature_mapping[
        ["category", "model_feature", "VUmc", "LUMC"]
    ]
    model_feature_mapping.to_csv(
        os.path.join(outputs_folder, cohort, "model_feature_mapping.csv"),
        index=False,
    )
    state["shap_data"] = shap_data

### Doc factors

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    state = feature_states[cohort]
    # Expand selected and free-text physician factors.
    doc_factors_data = analysis.construct_doc_factors_data_frame(
        predictions_and_outcomes
    )
    doc_factors_data = analysis.add_factor_labels(doc_factors_data)

    doc_factors = doc_factors_data.assign(count=1).pivot_table(
        index=["patient_id", "icu_discharged_at"],
        columns=["factor_label"],
        values="count",
        aggfunc="sum",
    )

    print(
        "Physicians on average give",
        (~doc_factors.isnull()).sum(axis=1).mean(),
        "reason",
    )

    doc_factors_summary = (
        pd.DataFrame((~doc_factors.isnull()).sum())
        .reset_index()
        .rename(columns={0: "count", "factor_label": "factor"})[["factor", "count"]]
        .sort_values(by=["count", "factor"], ascending=[False, True])
    )

    doc_factors_summary.to_csv(
        os.path.join(outputs_folder, cohort, "doc_factors.csv"), index=False
    )

    # Get physician reasons with new categories
    doc_factors_data_new = analysis.get_new_doc_features(doc_factors_data)
    doc_factors_new = doc_factors_data_new.assign(count=1).pivot_table(
        index=["patient_id", "icu_discharged_at"],
        columns=["new_factors"],
        values="count",
        aggfunc="sum",
    )

    doc_factors_new_summary = (
        pd.DataFrame((~doc_factors_new.isnull()).sum())
        .reset_index()
        .rename(columns={0: "count", "new_factors": "factor_category"})[
            ["factor_category", "count"]
        ]
        .sort_values(by=["count", "factor_category"], ascending=[False, True])
    )

    doc_factors_new_summary["percentage"] = round(
        100 * doc_factors_new_summary["count"] / doc_factors_new_summary["count"].sum(),
        1,
    )

    doc_factors_new_summary.to_csv(
        os.path.join(outputs_folder, cohort, "doc_factor_categories.csv"),
        index=False,
    )

    doc_factor_mapping = (
        doc_factors_data_new.assign(
            factor_display=risk_factors.physician_factor_display_labels(
                doc_factors_data_new["factor_label"]
            )
        )
        .groupby(["new_factors", "factor_display", "hospital"], dropna=False)
        .size()
        .unstack("hospital", fill_value=0)
        .reindex(columns=cohorts.HOSPITALS, fill_value=0)
        .rename_axis(index=["category", "physician_factor"], columns=None)
        .reset_index()
    )
    doc_factor_mapping["category"] = pd.Categorical(
        doc_factor_mapping["category"],
        categories=risk_factors.CATEGORIES,
        ordered=True,
    )
    doc_factor_mapping = doc_factor_mapping.sort_values(
        ["category", "physician_factor"]
    ).reset_index(drop=True)
    doc_factor_mapping = doc_factor_mapping[
        ["category", "physician_factor", "VUmc", "LUMC"]
    ]
    doc_factor_mapping.to_csv(
        os.path.join(outputs_folder, cohort, "physician_factor_mapping.csv"),
        index=False,
    )
    state["doc_factors_data_new"] = doc_factors_data_new

### Algorithm feature coverage
Calculates feature coverage based on the sum of absolute or raw Shapley values

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    state = feature_states[cohort]
    shap_data = state["shap_data"]
    doc_factors_data_new = state["doc_factors_data_new"]
    coverage_data = analysis.create_shap_coverage_data(shap_data, doc_factors_data_new)
    coverage_data.to_csv(
        os.path.join(outputs_folder, cohort, "feature_coverage.csv"),
        index=False,
    )

## Calibration figure summaries
Uses the canonical n_bootstraps and random_seed parameters and the shared run-scoped patient draws and combination fits. This cell collects curve summaries from those samples; it does not redraw patients or repeat identical model fits. Figure rendering reads saved summaries only.

In [ ]:
for cohort in cohorts.COHORT_ORDER:
    predictions_and_outcomes = analysis_cohorts[cohort]
    from algo_vs_doc import calibration

    calibration_curves, calibration_deciles, calibration_histograms = (
        calibration.plot_tables(
            predictions_and_outcomes,
            "outcome",
            {
                "Physician (Percentage)": "doc_prediction",
                "Model": "algo_prediction_at_doc_prediction",
                "Combined (Percentage)": "algo_and_doc_prediction",
                "Combined (Category)": "algo_and_doc_cat_prediction",
            },
            cluster_column="patient_cluster_id",
            n_bootstraps=n_bootstraps,
            random_seed=random_seed,
            combination_specs={
                "algo_and_doc_prediction": {
                    "prediction1": "doc_prediction",
                    "prediction2": "algo_prediction_at_doc_prediction",
                },
                "algo_and_doc_cat_prediction": {
                    "prediction1": "doc_prediction_cat",
                    "prediction2": "algo_prediction_at_doc_prediction",
                    "prediction1_is_ordinal": True,
                },
            },
        )
    )
    for name, summary in (
        ("calibration_curves", calibration_curves),
        ("calibration_deciles", calibration_deciles),
        ("calibration_histograms", calibration_histograms),
    ):
        summary.to_csv(
            os.path.join(plot_data_folder, cohort, name + ".csv"), index=False
        )